In [ ]:
mapper_path = "INSERT YOURS"

In [ ]:
import torch

all_mappers = [
    torch.load(mapper_path + f"mapper_lookahead_{i}.pt")
    for i in range(1, 6)
]

In [ ]:
from tqdm import trange
import torch

def get_projector(Vh, r):
    Vh[r:] = 0
    return Vh.adjoint() @ Vh

def get_projections(Vh, E, r):
    return get_projector(Vh, r) @ E

def get_cos(Vh, E, r):
    Ehat = get_projections(Vh, E, r)
    cos =  torch.nn.functional.cosine_similarity(E, Ehat, dim=0)
    del Ehat
    return cos

def get_all_cos(W, E):
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    del U
    result = torch.zeros((S.size(0) + 1, E.size(1)), dtype=E.dtype, device="cpu")
    for r in trange(S.size(0), 0, -1):
        result[r] = get_cos(Vh, E, r).cpu()
    return result

In [ ]:
mapper_matrices = [
    mapper["projection.weight"] for mapper in all_mappers
]

In [ ]:
from sae_lens import SAE

device = "cuda"

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",  # <- Release name
    sae_id="layer_15/width_16k/canonical",  # <- SAE id (not always a hook point!)
    device=device,
)

sae_matrix = sae.W_enc
sae_matrix.shape

In [ ]:
with torch.no_grad():
    cos_results = [
        get_all_cos(mapper_mat, sae_matrix)
        for mapper_mat in mapper_matrices
    ]

In [ ]:
torch.save(cos_results, "INSERT YOURS")